# Imbalance-Aware ML for UPI Fraud Detection: A Comparative Classifier Study

## Notebook 2 — Preprocessing (02_preprocessing.ipynb)

This notebook prepares the UPI transaction dataset for machine learning.
Since all features are already numeric (V1-V28 are PCA-transformed), 
preprocessing here is cleaner and simpler.

**Key Steps:**
- Load and verify the prepared dataset
- Feature engineering — derive amount_log and is_night
- Separate features from target label
- Stratified 80/20 train/test split
- Scale features using StandardScaler (required for SVM)
- Save all outputs for downstream notebooks

**No encoding needed** — all 30 features are already numeric.
**No missing values** — confirmed in EDA.

# Imports

In [1]:
# Data manipulation 
import pandas as pd                        # dataframes
import numpy as np                         # numerical operations

# Preprocessing tools 
from sklearn.preprocessing import StandardScaler   # feature scaling for SVM
from sklearn.model_selection import train_test_split  # train/test split

# Saving objects to disk 
import pickle                              # serialize scaler to .pkl
import os                                  # file path operations
import warnings
warnings.filterwarnings('ignore')

# Base path — all files load/save here 
base = r'C:\Users\ishaa\OneDrive\Documents\Projects\Fraud detection(UPI)'

print("✓ All libraries imported")

✓ All libraries imported


# Load Dataset

In [2]:
# Load the prepared UPI transaction dataset 
df = pd.read_csv(os.path.join(base, 'upi_transactions.csv'))

# Quick verification 
print("── Dataset Loaded ──")
print(f"  Shape         : {df.shape}")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"  Fraud cases   : {df['Class'].sum():,} ({df['Class'].mean()*100:.3f}%)")
print(f"  Legit cases   : {(df['Class']==0).sum():,}")
print(f"  Imbalance     : {(df['Class']==0).sum()//df['Class'].sum()}:1")

# Show column list 
print(f"\n── Columns ({df.shape[1]}) ──")
print(f"  PCA features  : V1-V28")
print(f"  Named features: Amount, transaction_hour")
print(f"  Target        : Class")

── Dataset Loaded ──
  Shape         : (15492, 31)
  Missing values: 0
  Fraud cases   : 492 (3.176%)
  Legit cases   : 15,000
  Imbalance     : 30:1

── Columns (31) ──
  PCA features  : V1-V28
  Named features: Amount, transaction_hour
  Target        : Class


# Feature Engineering
Creates two new features justified by EDA findings. **amount_log** handles the severe right skew in transaction amounts — without this transformation, the large outlier amounts (up to ₹3.5 lakh) would dominate distance-based calculations. **is_night** directly captures the EDA finding that fraud rate at 2AM (23.46%) is 7.4x higher than the overall average. The validation check at the end confirms the feature is actually useful — night transactions should and do show higher fraud rate.

In [3]:
# Engineer 2 new features from existing columns 

# Feature 1: amount_log 
# Transaction amounts are heavily right-skewed
# Most transactions are small (median ₹1,134) but max is ₹3,51,594
# log1p = log(1 + x) — compresses the scale safely
# Handles zero-amount transactions without producing -infinity
df['amount_log'] = np.log1p(df['Amount'])

# Feature 2: is_night 
# EDA showed fraud peaks at 2AM (23.46% fraud rate)
# Hours 0-5 are consistently above average fraud rate
# Binary flag: 1 if transaction occurs between midnight and 5AM
df['is_night'] = df['transaction_hour'].apply(
    lambda h: 1 if h <= 5 else 0
)

# Verify new features 
print("── New Features Created ──")
print(f"\n  amount_log samples : {df['amount_log'].head(5).round(3).tolist()}")
print(f"  amount_log range   : [{df['amount_log'].min():.3f}, {df['amount_log'].max():.3f}]")

print(f"\n  is_night distribution:")
print(f"    Night (1) : {df['is_night'].sum():,} transactions")
print(f"    Day   (0) : {(df['is_night']==0).sum():,} transactions")

# Check is_night vs fraud 
# Quick validation — night transactions should have higher fraud rate
night_fraud_rate = df[df['is_night']==1]['Class'].mean() * 100
day_fraud_rate   = df[df['is_night']==0]['Class'].mean() * 100
print(f"\n  Night fraud rate : {night_fraud_rate:.2f}%")
print(f"  Day fraud rate   : {day_fraud_rate:.2f}%")
print(f"  Night is {night_fraud_rate/day_fraud_rate:.1f}x more fraudulent than day")

print(f"\n  Total columns now: {df.shape[1]}")

── New Features Created ──

  amount_log samples : [4.007, 6.91, 8.793, 9.097, 7.342]
  amount_log range   : [0.000, 12.770]

  is_night distribution:
    Night (1) : 1,383 transactions
    Day   (0) : 14,109 transactions

  Night fraud rate : 8.97%
  Day fraud rate   : 2.61%
  Night is 3.4x more fraudulent than day

  Total columns now: 33


# Separate Features & Target
Separates the dataset into features (X) and target (y). Raw Amount is dropped because amount_log already captures all its information in a better-scaled form — keeping both would be redundant. Both transaction_hour and is_night are kept because they capture different things — the former gives the model the exact hour while the latter gives a simple binary fraud-risk flag.

In [4]:
# Define features (X) and target (y) 
# X = everything the model uses to make predictions
# y = what we are trying to predict (0=legit, 1=fraud)

# Drop Class column from features 
# Also drop raw Amount — amount_log already captures this info
# Keep transaction_hour AND is_night — they capture different things
# transaction_hour: specific hour (0-23) — granular temporal info
# is_night: binary flag (0/1) — captures the fraud-prone window
X = df.drop(columns=['Class', 'Amount'])  # drop target and raw amount
y = df['Class']                            # target label

print(f"── Features & Target ──")
print(f"  X shape : {X.shape}  (features)")
print(f"  y shape : {y.shape}  (target)")

print(f"\n── Feature List ({X.shape[1]} features) ──")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n── Target Distribution ──")
print(f"  Legitimate (0) : {(y==0).sum():,} ({(y==0).mean()*100:.3f}%)")
print(f"  Fraud      (1) : {(y==1).sum():,} ({(y==1).mean()*100:.3f}%)")

── Features & Target ──
  X shape : (15492, 31)  (features)
  y shape : (15492,)  (target)

── Feature List (31 features) ──
   1. V1
   2. V2
   3. V3
   4. V4
   5. V5
   6. V6
   7. V7
   8. V8
   9. V9
  10. V10
  11. V11
  12. V12
  13. V13
  14. V14
  15. V15
  16. V16
  17. V17
  18. V18
  19. V19
  20. V20
  21. V21
  22. V22
  23. V23
  24. V24
  25. V25
  26. V26
  27. V27
  28. V28
  29. transaction_hour
  30. amount_log
  31. is_night

── Target Distribution ──
  Legitimate (0) : 15,000 (96.824%)
  Fraud      (1) : 492 (3.176%)


# Train/Test Split
Splits the data into training (80% = 12,393 rows) and test (20% = 3,099 rows) sets with stratification. The fraud rate should be identical in both sets — approximately 3.176%. This is the most critical step in the entire preprocessing pipeline — the test set must never be touched by any resampling, scaling fitted on training data, or any other transformation that could cause information leakage.

In [5]:
# Stratified 80/20 train/test split 
# stratify=y — CRITICAL for imbalanced data
# Ensures both train and test sets maintain the same 3.176% fraud rate
# Without stratify, random split could place very few fraud cases
# in the test set, making evaluation unreliable

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,          # 20% test, 80% train
    random_state = 42,           # fixed seed — reproducible results
    stratify     = y             # preserve fraud ratio in both sets
)

# Verify split 
print("── Train/Test Split ──")
print(f"  Training set : {len(X_train):,} rows ({len(X_train)/len(X)*100:.0f}%)")
print(f"  Test set     : {len(X_test):,} rows  ({len(X_test)/len(X)*100:.0f}%)")

print(f"\n── Fraud Rate Preserved ──")
print(f"  Full dataset : {y.mean()*100:.3f}%")
print(f"  Training set : {y_train.mean()*100:.3f}%")
print(f"  Test set     : {y_test.mean()*100:.3f}%")
print(f"\n  Train fraud cases : {y_train.sum():,}")
print(f"  Test fraud cases  : {y_test.sum():,}")
print(f"\n✓ Stratification confirmed — fraud rate consistent across splits")

# Important reminder 
print(f"\n⚠️  SMOTE applied to X_train ONLY in 03_imbalance_handling")
print(f"    X_test stays untouched — real-world distribution preserved")

── Train/Test Split ──
  Training set : 12,393 rows (80%)
  Test set     : 3,099 rows  (20%)

── Fraud Rate Preserved ──
  Full dataset : 3.176%
  Training set : 3.179%
  Test set     : 3.162%

  Train fraud cases : 394
  Test fraud cases  : 98

✓ Stratification confirmed — fraud rate consistent across splits

⚠️  SMOTE applied to X_train ONLY in 03_imbalance_handling
    X_test stays untouched — real-world distribution preserved


# Scale Features
Scales all features to zero mean and unit variance using StandardScaler. Although V1-V28 are already roughly standardized from PCA, amount_log and transaction_hour are on completely different scales — this would cause SVM to effectively ignore the PCA features. The scaler is fitted only on training data and then applied to both train and test — fitting on test would leak test statistics and artificially improve results.

In [6]:
# StandardScaler — required for SVM 
# SVM calculates distances between data points
# Features at different scales dominate the distance calculation
# V1-V28 are already approximately standardized (PCA output)
# But Amount_log and transaction_hour are on very different scales
# StandardScaler transforms all features to mean=0, std=1

scaler = StandardScaler()

# CRITICAL: Fit ONLY on training data 
# fit_transform on train: learns mean/std from train, applies scaling
# transform on test: applies SAME mean/std learned from train
# Fitting on test data would leak test statistics into the scaler
# — a subtle but serious form of data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)    # no fit — transform only

# Convert back to DataFrames 
# Preserves column names for readability in downstream notebooks
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X_test.columns)

# Verify scaling 
print("── Scaling Verification ──")
print(f"  Mean of scaled features (should be ~0):")
print(X_train_scaled.mean().round(4).tail(5))  # show last 5 — non-PCA features
print(f"\n  Std of scaled features (should be ~1):")
print(X_train_scaled.std().round(4).tail(5))

print(f"\n✓ Scaling complete")
print(f"  Scaled versions → SVM only")
print(f"  Unscaled versions → Random Forest and XGBoost")

── Scaling Verification ──
  Mean of scaled features (should be ~0):
V27                 0.0
V28                -0.0
transaction_hour   -0.0
amount_log          0.0
is_night           -0.0
dtype: float64

  Std of scaled features (should be ~1):
V27                 1.0
V28                 1.0
transaction_hour    1.0
amount_log          1.0
is_night            1.0
dtype: float64

✓ Scaling complete
  Scaled versions → SVM only
  Unscaled versions → Random Forest and XGBoost


# Save All Outputs

In [7]:
# Save all train/test splits 
# Unscaled — Random Forest and XGBoost
X_train.to_csv(os.path.join(base, 'X_train.csv'), index=False)
X_test.to_csv(os.path.join(base, 'X_test.csv'),   index=False)
y_train.to_csv(os.path.join(base, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(base, 'y_test.csv'),   index=False)
print("✓ Unscaled train/test sets saved")

# Scaled — SVM only
X_train_scaled.to_csv(os.path.join(base, 'X_train_scaled.csv'), index=False)
X_test_scaled.to_csv(os.path.join(base, 'X_test_scaled.csv'),   index=False)
print("✓ Scaled train/test sets saved")

# Save scaler 
# Must save the fitted scaler so evaluation notebook applies
# the EXACT same transformation as training notebook
models_path = os.path.join(base, 'models')
os.makedirs(models_path, exist_ok=True)

with open(os.path.join(models_path, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print("✓ Scaler saved → models/scaler.pkl")

# Final summary 
print(f"""
══════════════════════════════════════════════
  PREPROCESSING COMPLETE
══════════════════════════════════════════════
  Total rows          : {len(df):,}
  Final features      : {X_train.shape[1]}
  Training samples    : {len(X_train):,}
  Test samples        : {len(X_test):,}
  Train fraud cases   : {y_train.sum():,}
  Test fraud cases    : {y_test.sum():,}
  Files saved         : 6 CSVs + 1 scaler.pkl
══════════════════════════════════════════════
  → Next: 03_imbalance_handling
══════════════════════════════════════════════
""")

✓ Unscaled train/test sets saved
✓ Scaled train/test sets saved
✓ Scaler saved → models/scaler.pkl

══════════════════════════════════════════════
  PREPROCESSING COMPLETE
══════════════════════════════════════════════
  Total rows          : 15,492
  Final features      : 31
  Training samples    : 12,393
  Test samples        : 3,099
  Train fraud cases   : 394
  Test fraud cases    : 98
  Files saved         : 6 CSVs + 1 scaler.pkl
══════════════════════════════════════════════
  → Next: 03_imbalance_handling
══════════════════════════════════════════════



## Preprocessing Summary

This notebook successfully prepared the UPI transaction dataset for machine 
learning with minimal transformations — a direct benefit of starting with a 
clean, fully numeric dataset.

### Steps Completed

- Loaded 15,492 UPI transactions with 31 original columns and zero missing values
- Engineered 2 new features:
  - `amount_log` — log-transformed Amount (range: 0.000 to 12.770) to handle 
    right skew in transaction amounts
  - `is_night` — binary flag for hours 0–5, capturing the peak fraud window 
    identified in EDA (8.97% night fraud rate vs 2.61% day — 3.4x difference)
- Dropped raw `Amount` — fully replaced by `amount_log`
- Final feature set: 31 features (V1–V28 + transaction_hour + amount_log + is_night)
- No label encoding required — all features already numeric
- No missing value imputation — zero missing values confirmed

### Train/Test Split Results

| Set | Rows | Fraud Cases | Fraud Rate |
|---|---|---|---|
| Training (80%) | 12,393 | 394 | 3.179% |
| Test (20%) | 3,099 | 98 | 3.162% |
| **Total** | **15,492** | **492** | **3.176%** |

Stratified splitting preserved the fraud rate consistently across both sets — 
training (3.179%) and test (3.162%) are virtually identical to the full dataset 
rate (3.176%).

### Key Decisions & Justifications

| Decision | Justification |
|---|---|
| Kept `transaction_hour` AND `is_night` | Capture different temporal signals — hour gives granular info, is_night gives fraud-risk flag |
| Dropped raw `Amount` | `amount_log` is a better-scaled replacement — eliminates right skew |
| No encoding | All 31 features already numeric — V1-V28 are PCA-transformed |
| Scaled for SVM only | RF and XGBoost are scale-invariant tree-based models |
| Fit scaler on train only | Prevents test set statistics leaking into the scaler |

### Scaling Verification

StandardScaler confirmed — all features scaled to mean ≈ 0 and std ≈ 1 on 
training data. The same transformation applied to test data using training 
statistics only.

**→ Next: 03_imbalance_handling.ipynb** — Compare Baseline, SMOTE, 
ADASYN, and SMOTE+Tomek strategies on the training set only.